# Olist 电商经营分析：销售增长、客户购买行为与履约体验诊断

**核心业务问题**：平台的商品销售金额增长来自哪里？客户在首次成功购买后90天内是否再次购买？配送履约与客户评价是否存在值得优化的问题？

- **数据来源**：Olist Brazilian E-Commerce Public Dataset（7张业务表）。

> **分析口径**：经营指标聚焦最终状态为 `delivered` 的订单；商品销售金额为 `order_items.price` 合计，不含运费、成本、退款，也不等于财务确认的净收入。月度分析按**下单日期**分组，而不是按送达日期分组。

## 0. 数据字典与指标口径

| 数据表 | 内容 | 一行的粒度 | 重要字段 |
|---|---|---|---|
| `orders` | 订单及履约时间 | 一笔订单 | `order_id`, `customer_id`, `order_status`, `order_purchase_timestamp`, `order_delivered_customer_date`, `order_estimated_delivery_date` |
| `customers` | 客户及地理信息 | 一条客户订单记录 | `customer_id`, `customer_unique_id`, `customer_state` |
| `order_items` | 订单商品明细 | 一条订单商品记录 | `order_id`, `product_id`, `price`, `freight_value` |
| `products` | 商品信息 | 一个商品 | `product_id`, `product_category_name` |
| `categories` | 商品品类翻译 | 一个葡语品类名称 | `product_category_name`, `product_category_name_english` |
| `payments` | 支付明细 | 一笔支付记录 | `order_id`, `payment_value`, `payment_type` |
| `reviews` | 评价信息 | 一条评价记录 | `order_id`, `review_score` |

**容易混淆的字段**：`customer_id` 是与订单关联的客户记录标识；分析独立客户和复购，应通过它连接到 `customers.customer_unique_id`。一笔订单可能有多条商品、支付或评价记录，连接明细表时必须关注**重复行导致的金额放大**。

**核心指标**：

| 指标 | 计算口径 |
|---|---|
| 已送达订单数 | `orders.order_status = 'delivered'` 的不同 `order_id` 数 |
| 商品销售金额（BRL） | 已送达订单 `SUM(order_items.price)`，不包含运费 |
| 平均每单商品金额 | 商品销售金额 ÷ 已送达订单数 |
| 重复购买客户占比 | 观察到至少2笔已送达订单的独立客户 ÷ 至少1笔已送达订单的独立客户 |
| 90天重复购买率 | 有完整90天观察期的客户中，首次符合条件订单下单后90天内**再次下单且第二单最终已送达**的客户比例 |

> **限制**：数据的最早订单未必是客户历史真正首单；使用已送达订单中的最晚下单时间作为观察截止点，是基于当前公开数据的保守处理，并不能消除全部数据覆盖或订单截断问题。不同月份客户不是随机分组，Cohort差异不意味着因果关系。

## 1. 数据读取与质量检查

### 1.1 导入7张核心业务表

先检查文件是否齐全，随后将数据读入Pandas。

In [1]:
from pathlib import Path
import pandas as pd
import sqlite3

# 在Notebook输出中完整展示月度及品类分析列
pd.set_option("display.max_columns", None)

DATA_DIR = Path("data")
file_map = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "categories": "product_category_name_translation.csv",
}

missing = [filename for filename in file_map.values()
           if not (DATA_DIR / filename).exists()]
assert not missing, f"缺少CSV文件，请检查data/目录：{missing}"

T = {table: pd.read_csv(DATA_DIR / filename)
     for table, filename in file_map.items()}

table_sizes = pd.DataFrame([
    {"table": name, "rows": len(df), "columns": df.shape[1]}
    for name, df in T.items()
])
display(table_sizes)

,table,rows,columns
0,customers,99441,5
1,orders,99441,8
2,order_items,112650,7
3,payments,103886,5
4,reviews,99224,7
5,products,32951,9
6,categories,71,2


### 1.2 验证主键、表关系与核心字段缺失情况

In [2]:
checks = {
    "orders.order_id唯一": T["orders"]["order_id"].is_unique,
    "customers.customer_id唯一": T["customers"]["customer_id"].is_unique,
    "每笔订单能找到客户记录": T["orders"]["customer_id"].isin(T["customers"]["customer_id"]).all(),
    "每条商品明细能找到订单": T["order_items"]["order_id"].isin(T["orders"]["order_id"]).all(),
    "每条支付记录能找到订单": T["payments"]["order_id"].isin(T["orders"]["order_id"]).all(),
    "每条评价记录能找到订单": T["reviews"]["order_id"].isin(T["orders"]["order_id"]).all(),
    "每条商品明细能找到商品": T["order_items"]["product_id"].isin(T["products"]["product_id"]).all(),
}
for label, passed in checks.items():
    print(("✅ " if passed else "⚠️ ") + label)

print("\n多条明细对应同一订单的比例：")
for name in ["order_items", "payments", "reviews"]:
    cnt = T[name].groupby("order_id").size()
    print(f"{name}: {cnt.gt(1).mean():.2%}")

print("\n关键字段缺失值：")
for table, column in [
    ("orders", "order_purchase_timestamp"),
    ("orders", "order_delivered_customer_date"),
    ("orders", "order_estimated_delivery_date"),
    ("customers", "customer_unique_id"),
    ("order_items", "price"),
    ("reviews", "review_score"),
]:
    print(f"{table}.{column}: {T[table][column].isna().sum():,}")

assert all(checks.values()), "关联检查未全部通过，请先检查数据"

✅ orders.order_id唯一
✅ customers.customer_id唯一
✅ 每笔订单能找到客户记录
✅ 每条商品明细能找到订单
✅ 每条支付记录能找到订单
✅ 每条评价记录能找到订单
✅ 每条商品明细能找到商品

多条明细对应同一订单的比例：
order_items: 9.94%
payments: 2.98%
reviews: 0.55%

关键字段缺失值：
orders.order_purchase_timestamp: 0
orders.order_delivered_customer_date: 2,965
orders.order_estimated_delivery_date: 0
customers.customer_unique_id: 0
order_items.price: 0
reviews.review_score: 0


### 1.3 初始化SQLite，并核对订单状态

订单状态并不都代表交易已完成。后续销售和复购使用 `delivered` 口径。

In [3]:
conn = sqlite3.connect(":memory:")
for name, df in T.items():
    df.to_sql(name, conn, if_exists="replace", index=False)

# 为后续多表关联和复购查询添加索引
conn.execute("CREATE INDEX idx_orders_customer_id ON orders(customer_id)")
conn.execute("CREATE INDEX idx_customers_id ON customers(customer_id)")
conn.execute("CREATE INDEX idx_orders_order_id ON orders(order_id)")
conn.execute("CREATE INDEX idx_items_order_id ON order_items(order_id)")

status_summary = pd.read_sql_query("""
SELECT order_status, COUNT(*) AS order_count
FROM orders
GROUP BY order_status
ORDER BY order_count DESC;
""", conn)
display(status_summary)

print("已送达订单最早/最晚下单时间：")
display(pd.read_sql_query("""
SELECT MIN(order_purchase_timestamp) AS first_purchase_time,
       MAX(order_purchase_timestamp) AS last_purchase_time
FROM orders WHERE order_status = 'delivered';
""", conn))

,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


已送达订单最早/最晚下单时间：


,first_purchase_time,last_purchase_time
0,2016-09-15 12:16:38,2018-08-29 15:00:37


## 2. 销售增长与品类贡献分析

### 2.1 核心经营指标 + 月度趋势 + 环比增长拆解

业务问题：**月度销售金额为什么变化？是订单数量，还是平均每单金额？**

> 月度主要展示2017-02至2018-08：2016年部分月份记录稀疏；不能把中间缺月时的上一条记录误认为相邻自然月。

In [4]:
sql_monthly = """
WITH item_by_order AS (
    SELECT order_id, SUM(price) AS item_amount
    FROM order_items
    GROUP BY order_id
),
monthly AS (
    SELECT
        SUBSTR(o.order_purchase_timestamp, 1, 7) AS order_month,
        COUNT(*) AS order_count,
        SUM(i.item_amount) AS total_sales,
        SUM(i.item_amount) * 1.0 / COUNT(*) AS avg_order_amount
    FROM orders AS o
    LEFT JOIN item_by_order AS i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp IS NOT NULL
    GROUP BY SUBSTR(o.order_purchase_timestamp, 1, 7)
),
monthly_prev AS (
    SELECT *,
        LAG(total_sales) OVER (ORDER BY order_month) AS prev_sales,
        LAG(order_count) OVER (ORDER BY order_month) AS prev_orders,
        LAG(avg_order_amount) OVER (ORDER BY order_month) AS prev_avg_amount
    FROM monthly
)
SELECT
    order_month, order_count,
    ROUND(total_sales, 2) AS total_sales,
    ROUND(avg_order_amount, 2) AS avg_order_amount,
    ROUND((total_sales - prev_sales) * 100.0 / NULLIF(prev_sales, 0), 2)
        AS sales_mom_pct,
    ROUND((order_count - prev_orders) * 100.0 / NULLIF(prev_orders, 0), 2)
        AS order_mom_pct,
    ROUND((avg_order_amount - prev_avg_amount) * 100.0
          / NULLIF(prev_avg_amount, 0), 2) AS avg_order_mom_pct
FROM monthly_prev
ORDER BY order_month;
"""
monthly_sales = pd.read_sql_query(sql_monthly, conn)

# 一份结果同时提供平台总KPI、月度趋势与11月增长拆解
kpi_result = pd.DataFrame([{
    "delivered_orders": int(monthly_sales["order_count"].sum()),
    "total_item_amount": round(monthly_sales["total_sales"].sum(), 2),
    "avg_item_amount_per_order": round(
        monthly_sales["total_sales"].sum() / monthly_sales["order_count"].sum(), 2
    ),
}])
monthly_analysis = monthly_sales.loc[
    monthly_sales["order_month"].between("2017-02", "2018-08")
].copy()
growth_analysis = monthly_analysis.loc[
    monthly_analysis["order_month"] == "2017-11",
    ["order_month", "sales_mom_pct", "order_mom_pct", "avg_order_mom_pct"]
]

print("① 核心经营指标：")
display(kpi_result)
print("② 月度趋势及3个环比指标：")
display(monthly_analysis)
print("③ 2017年11月增长拆解：")
display(growth_analysis)

# 数据核对：月度订单数应与订单状态汇总一致
assert int(kpi_result.loc[0, "delivered_orders"]) == int(
    status_summary.loc[status_summary["order_status"] == "delivered", "order_count"].iloc[0]
)
print("✅ 月度订单数与状态汇总一致")

① 核心经营指标：


,delivered_orders,total_item_amount,avg_item_amount_per_order
0,96478,13221498.11,137.04


② 月度趋势及3个环比指标：


,order_month,order_count,total_sales,avg_order_amount,sales_mom_pct,order_mom_pct,avg_order_mom_pct
4,2017-02,1653,234223.40,141.70,109.51,120.40,-4.94
5,2017-03,2546,359198.85,141.08,53.36,54.02,-0.43
6,2017-04,2303,340669.68,147.92,-5.16,-9.54,4.85
7,2017-05,3546,489338.25,138.00,43.64,53.97,-6.71
8,2017-06,3135,421923.37,134.58,-13.78,-11.59,-2.47
9,2017-07,3872,481604.52,124.38,14.15,23.51,-7.58
10,2017-08,4193,554699.70,132.29,15.18,8.29,6.36
11,2017-09,4150,607399.67,146.36,9.50,-1.03,10.64
12,2017-10,4478,648247.65,144.76,6.73,7.90,-1.09
13,2017-11,7289,987765.37,135.51,52.37,62.77,-6.39


③ 2017年11月增长拆解：


,order_month,sales_mom_pct,order_mom_pct,avg_order_mom_pct
13,2017-11,52.37,62.77,-6.39


✅ 月度订单数与状态汇总一致


**阶段发现**：2017年11月商品销售额环比 **+52.37%**，订单数量 **+62.77%**，平均每单商品金额 **−6.39%**。因此销售增长主要是订单数量扩大推动，而不是每单金额上升；数据不足以确认是否由促销活动造成。

### 2.2 2017年11月主要品类增长贡献

业务问题：**哪些品类贡献了销售金额的净增长？**

同一份SQL同时计算10月和11月的品类订单数、商品金额、销售增量及对平台净增长的贡献比例。品类订单数采用 `COUNT(DISTINCT order_id)`；跨品类的订单数不能相加当成全平台订单数。

In [5]:
sql_category = """
WITH category_month AS (
    SELECT
        SUBSTR(o.order_purchase_timestamp, 1, 7) AS order_month,
        COALESCE(c.product_category_name_english,
                 p.product_category_name, 'unknown') AS category,
        COUNT(DISTINCT o.order_id) AS order_count,
        SUM(i.price) AS sales
    FROM orders AS o
    JOIN order_items AS i ON o.order_id = i.order_id
    LEFT JOIN products AS p ON i.product_id = p.product_id
    LEFT JOIN categories AS c
        ON p.product_category_name = c.product_category_name
    WHERE o.order_status = 'delivered'
      AND SUBSTR(o.order_purchase_timestamp, 1, 7) IN ('2017-10', '2017-11')
    GROUP BY order_month, category
),
category_compare AS (
    SELECT category,
        SUM(CASE WHEN order_month = '2017-10' THEN order_count ELSE 0 END)
            AS orders_oct,
        SUM(CASE WHEN order_month = '2017-11' THEN order_count ELSE 0 END)
            AS orders_nov,
        SUM(CASE WHEN order_month = '2017-10' THEN sales ELSE 0 END)
            AS sales_oct,
        SUM(CASE WHEN order_month = '2017-11' THEN sales ELSE 0 END)
            AS sales_nov
    FROM category_month
    GROUP BY category
)
SELECT category, orders_oct, orders_nov,
    ROUND(sales_oct, 2) AS sales_oct,
    ROUND(sales_nov, 2) AS sales_nov,
    ROUND(sales_nov - sales_oct, 2) AS sales_increase,
    ROUND(100.0 * (sales_nov - sales_oct) /
          NULLIF((SELECT SUM(sales_nov - sales_oct) FROM category_compare), 0), 2)
        AS contribution_pct
FROM category_compare
ORDER BY sales_increase DESC;
"""
category_growth = pd.read_sql_query(sql_category, conn)
category_top10 = category_growth.head(10).copy()

# 用所有品类的净增量做分母，避免仅用Top 10计算贡献率
total_increase = round(category_growth["sales_increase"].sum(), 2)
top3_increase = round(category_top10.head(3)["sales_increase"].sum(), 2)
top3_contribution = 100 * top3_increase / total_increase

print("商品销售金额增量Top 10品类：")
display(category_top10)
print(f"全平台商品金额净增量：R${total_increase:,.2f}")
print(f"Top 3品类金额增量：R${top3_increase:,.2f}")
print(f"Top 3品类贡献率：{top3_contribution:.2f}%")

platform_growth = (
    monthly_sales.loc[monthly_sales["order_month"] == "2017-11", "total_sales"].iloc[0]
    - monthly_sales.loc[monthly_sales["order_month"] == "2017-10", "total_sales"].iloc[0]
)
assert abs(total_increase - platform_growth) < 0.02, "品类净增量未对齐平台净增量"
print("✅ 品类净增量与平台销售金额变化一致")

商品销售金额增量Top 10品类：


,category,orders_oct,orders_nov,sales_oct,sales_nov,sales_increase,contribution_pct
0,bed_bath_table,442,804,46007.70,87957.63,41949.93,12.36
1,health_beauty,322,519,40698.99,78274.40,37575.41,11.07
2,furniture_decor,266,536,30009.74,62091.27,32081.53,9.45
3,watches_gifts,282,424,64874.63,95292.34,30417.71,8.96
4,toys,272,443,33324.42,62611.26,29286.84,8.63
5,computers_accessories,253,431,42009.38,69676.32,27666.94,8.15
6,garden_tools,213,430,25798.75,44275.18,18476.43,5.44
7,housewares,188,345,15883.68,33872.42,17988.74,5.30
8,perfumery,175,287,16877.82,31612.13,14734.31,4.34
9,sports_leisure,417,542,48568.98,62686.24,14117.26,4.16


全平台商品金额净增量：R$339,517.72
Top 3品类金额增量：R$111,606.87
Top 3品类贡献率：32.87%
✅ 品类净增量与平台销售金额变化一致


**阶段发现**：家居床品（`bed_bath_table`）、健康美容（`health_beauty`）、家具装饰（`furniture_decor`）合计贡献2017年11月商品销售金额净增量的约 **32.87%**，是销售金额贡献。

## 3. 客户购买行为与90天复购分析

### 3.1 购买次数分布 + 重复购买客户占比

业务问题：观察期内，多大比例的独立客户不止一次完成购买？这里按 `customer_unique_id` 计算

In [6]:
sql_frequency = """
WITH customer_orders AS (
    SELECT c.customer_unique_id,
           COUNT(DISTINCT o.order_id) AS order_count
    FROM orders AS o
    JOIN customers AS c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id
)
SELECT order_count, COUNT(*) AS customer_count
FROM customer_orders
GROUP BY order_count
ORDER BY order_count;
"""
purchase_frequency = pd.read_sql_query(sql_frequency, conn)

customers_total = int(purchase_frequency["customer_count"].sum())
one_time_customers = int(purchase_frequency.loc[
    purchase_frequency["order_count"] == 1, "customer_count"].sum())
repeat_customers_all_period = customers_total - one_time_customers
repeat_customer_share = repeat_customers_all_period / customers_total * 100

print("购买次数分布：")
display(purchase_frequency)
print(f"已送达订单购买客户数：{customers_total:,}")
print(f"只购买1次的客户数：{one_time_customers:,}")
print(f"购买至少2次的客户数：{repeat_customers_all_period:,}")
print(f"重复购买客户占比（全观察期）：{repeat_customer_share:.2f}%")

checked_orders = int((purchase_frequency["order_count"] *
                     purchase_frequency["customer_count"]).sum())
assert checked_orders == int(kpi_result.loc[0, "delivered_orders"])
print(f"✅ 客户购买次数汇总核对：{checked_orders:,}笔已送达订单")

购买次数分布：


,order_count,customer_count
0,1,90557
1,2,2573
2,3,181
3,4,28
4,5,9
5,6,5
6,7,3
7,9,1
8,15,1


已送达订单购买客户数：93,358
只购买1次的客户数：90,557
购买至少2次的客户数：2,801
重复购买客户占比（全观察期）：3.00%
✅ 客户购买次数汇总核对：96,478笔已送达订单


### 3.2 90天复购率 + 按首次购买月份分组的Cohort

业务问题：**首次成功购买后90天内，有多少客户再次成功购买？不同首次购买月份是否存在差异？**

In [7]:
# 可重复运行的临时视图；避免重复编写两大段Cohort计算SQL
conn.execute("DROP VIEW IF EXISTS customer_repeat_90d")

conn.execute("""
CREATE TEMP VIEW customer_repeat_90d AS
WITH delivered_orders AS (
    SELECT c.customer_unique_id,
           o.order_purchase_timestamp AS purchase_time
    FROM orders AS o
    JOIN customers AS c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp IS NOT NULL
),
first_purchase AS (
    SELECT customer_unique_id, MIN(purchase_time) AS first_time
    FROM delivered_orders
    GROUP BY customer_unique_id
)
SELECT
    f.customer_unique_id,
    SUBSTR(f.first_time, 1, 7) AS cohort_month,
    MAX(CASE
        WHEN d.purchase_time > f.first_time
         AND d.purchase_time <= DATETIME(f.first_time, '+90 days')
        THEN 1 ELSE 0 END) AS repeat_90d
FROM first_purchase AS f
JOIN delivered_orders AS d
    ON f.customer_unique_id = d.customer_unique_id
WHERE f.first_time <= (
    SELECT DATETIME(MAX(purchase_time), '-90 days')
    FROM delivered_orders
)
GROUP BY f.customer_unique_id, f.first_time;
""")

repeat_90d_result = pd.read_sql_query("""
SELECT COUNT(*) AS eligible_customers,
       SUM(repeat_90d) AS repeat_customers,
       ROUND(AVG(repeat_90d) * 100, 2) AS repeat_rate_90d_pct
FROM customer_repeat_90d;
""", conn)

cohort_repeat = pd.read_sql_query("""
SELECT cohort_month,
       COUNT(*) AS customer_count,
       SUM(repeat_90d) AS repeat_customers,
       ROUND(AVG(repeat_90d) * 100, 2) AS repeat_rate_90d_pct
FROM customer_repeat_90d
WHERE cohort_month BETWEEN '2017-01' AND '2018-04'
GROUP BY cohort_month
ORDER BY cohort_month;
""", conn)

print("① 全平台90天复购率：")
display(repeat_90d_result)
print("② 按首次成功购买月份划分的同期群：")
display(cohort_repeat)
print("说明：两次订单都要求最终已送达，90天按下单日期判断；不是严格意义的终身复购率。")

① 全平台90天复购率：


,eligible_customers,repeat_customers,repeat_rate_90d_pct
0,75320,1514,2.01


② 按首次成功购买月份划分的同期群：


,cohort_month,customer_count,repeat_customers,repeat_rate_90d_pct
0,2017-01,717,22,3.07
1,2017-02,1628,26,1.60
2,2017-03,2503,56,2.24
3,2017-04,2256,42,1.86
4,2017-05,3451,86,2.49
5,2017-06,3037,77,2.54
6,2017-07,3752,82,2.19
7,2017-08,4057,95,2.34
8,2017-09,4004,100,2.50
9,2017-10,4328,86,1.99


说明：两次订单都要求最终已送达，90天按下单日期判断；不是严格意义的终身复购率。


### 3.3 客户分析阶段结论与局限

- 在全部观察期，**93,358** 名至少有1笔已送达订单的客户中，**2,801** 人有至少2笔已送达订单（**3.00%**）。
- 按完整90天观察窗口计算，**75,320** 名符合条件的客户中，**1,514** 人在首次成功购买后90天内再次下单、且新订单最终已送达（**2.01%**）。
- 2017年10月与11月首次成功购买的客户数分别为 **4,328** 和 **7,060**；其90天复购率分别为 **1.99%** 和 **2.03%**。新增购买客户规模扩大，但90天复购率未出现同等幅度的变化。

## 4. 物流履约与客户评价分析

**待回答的业务问题**：

1. 已送达订单中，有多少超出预计送达日期？延迟的天数分布如何？
2. 延迟交付订单的平均评价分数与低评分比例，是否与准时送达订单存在差异？
3. 是否存在延迟率较高且订单规模较大的州/品类，可作为优先排查对象？

### 4.1 检查不同订单状态下的配送日期缺失情况

In [8]:
import pandas as pd

In [9]:
sql = """
SELECT
    order_status,

    COUNT(*) AS order_count,

    SUM(
        CASE
            WHEN order_delivered_customer_date IS NULL
              OR TRIM(order_delivered_customer_date) = ''
            THEN 1 ELSE 0
        END
    ) AS missing_actual_date,

    SUM(
        CASE
            WHEN order_estimated_delivery_date IS NULL
              OR TRIM(order_estimated_delivery_date) = ''
            THEN 1 ELSE 0
        END
    ) AS missing_estimated_date,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN order_delivered_customer_date IS NULL
                  OR TRIM(order_delivered_customer_date) = ''
                THEN 1 ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS missing_actual_pct

FROM orders

GROUP BY order_status

ORDER BY order_count DESC;
"""

delivery_quality = pd.read_sql_query(sql, conn)

display(delivery_quality)


,order_status,order_count,missing_actual_date,missing_estimated_date,missing_actual_pct
0,delivered,96478,8,0,0.01
1,shipped,1107,1107,0,100.00
2,canceled,625,619,0,99.04
3,unavailable,609,609,0,100.00
4,invoiced,314,314,0,100.00
5,processing,301,301,0,100.00
6,created,5,5,0,100.00
7,approved,2,2,0,100.00


### 4.2 计算配送效率及准时送达率

In [10]:
sql = """
WITH delivery_base AS (
    SELECT
        order_id,

        JULIANDAY(order_delivered_customer_date)
        - JULIANDAY(order_purchase_timestamp)
        AS delivery_days,

        CASE
            WHEN DATE(order_delivered_customer_date)
               > DATE(order_estimated_delivery_date)
            THEN 1 ELSE 0
        END AS is_late

    FROM orders

    WHERE order_status = 'delivered'
      AND order_purchase_timestamp IS NOT NULL
      AND order_delivered_customer_date IS NOT NULL
      AND order_estimated_delivery_date IS NOT NULL

      AND JULIANDAY(order_delivered_customer_date)
          >= JULIANDAY(order_purchase_timestamp)
)

SELECT
    COUNT(*) AS valid_orders,

    ROUND(AVG(delivery_days), 2)
        AS avg_delivery_days,

    SUM(is_late) AS late_orders,

    ROUND(
        100.0 * SUM(is_late) / COUNT(*), 2
    ) AS late_rate_pct,

    ROUND(
        100.0 * SUM(1 - is_late) / COUNT(*), 2
    ) AS on_time_rate_pct

FROM delivery_base;
"""

delivery_performance = pd.read_sql_query(sql, conn)

display(delivery_performance)


,valid_orders,avg_delivery_days,late_orders,late_rate_pct,on_time_rate_pct
0,96470,12.56,6534,6.77,93.23


### 4.3 比较准时与延迟订单的客户评分

In [11]:
sql = """
WITH valid_delivery AS (
    SELECT
        order_id,

        CASE
            WHEN DATE(order_delivered_customer_date)
               > DATE(order_estimated_delivery_date)
            THEN 'late'
            ELSE 'on_time'
        END AS delivery_status

    FROM orders

    WHERE order_status = 'delivered'
      AND order_purchase_timestamp IS NOT NULL
      AND order_delivered_customer_date IS NOT NULL
      AND order_estimated_delivery_date IS NOT NULL
      AND JULIANDAY(order_delivered_customer_date)
          >= JULIANDAY(order_purchase_timestamp)
),

review_ranked AS (
    SELECT
        order_id,
        review_score,

        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY review_answer_timestamp DESC,
                     review_creation_date DESC,
                     review_id DESC
        ) AS rn

    FROM reviews
    WHERE review_score BETWEEN 1 AND 5
)

SELECT
    d.delivery_status,

    COUNT(*) AS reviewed_orders,

    ROUND(AVG(r.review_score), 2) AS avg_review_score,

    SUM(
        CASE WHEN r.review_score <= 2
             THEN 1 ELSE 0 END
    ) AS low_score_orders,

    ROUND(
        100.0 * SUM(
            CASE WHEN r.review_score <= 2
                 THEN 1 ELSE 0 END
        ) / COUNT(*),
        2
    ) AS low_score_pct

FROM valid_delivery AS d

JOIN review_ranked AS r
    ON d.order_id = r.order_id
   AND r.rn = 1

GROUP BY d.delivery_status

ORDER BY d.delivery_status;
"""

delivery_reviews = pd.read_sql_query(sql, conn)

display(delivery_reviews)


,delivery_status,reviewed_orders,avg_review_score,low_score_orders,low_score_pct
0,late,6381,2.27,3983,62.42
1,on_time,89443,4.29,8289,9.27


在Olist平台已送达且有有效评价的订单中，延迟送达订单的平均评分为2.27分，明显低于准时送达订单的4.29分。
同时，延迟订单1—2星低评分比例达到62.42%，约为准时订单的6.73倍。说明配送延迟与客户负面评价存在较强关联，值得作为客户体验优化的重点调查方向。

### 4.4 分析延迟天数与低评分比例的关系

In [12]:
sql = """
WITH delivery_base AS (
    SELECT
        order_id,
        CAST(
            JULIANDAY(DATE(order_delivered_customer_date))
            - JULIANDAY(DATE(order_estimated_delivery_date))
            AS INTEGER
        ) AS days_late
    FROM orders
    WHERE order_status = 'delivered'
      AND order_purchase_timestamp IS NOT NULL
      AND order_delivered_customer_date IS NOT NULL
      AND order_estimated_delivery_date IS NOT NULL
      AND JULIANDAY(order_delivered_customer_date)
          >= JULIANDAY(order_purchase_timestamp)
),

review_ranked AS (
    SELECT
        order_id,
        review_score,
        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY review_answer_timestamp DESC,
                     review_creation_date DESC,
                     review_id DESC
        ) AS rn
    FROM reviews
    WHERE review_score BETWEEN 1 AND 5
),

grouped_orders AS (
    SELECT
        CASE
            WHEN d.days_late <= 0 THEN '0_on_time'
            WHEN d.days_late <= 3 THEN '1_1-3_days'
            WHEN d.days_late <= 7 THEN '2_4-7_days'
            WHEN d.days_late <= 14 THEN '3_8-14_days'
            ELSE '4_15+_days'
        END AS delay_group,
        r.review_score
    FROM delivery_base AS d
    JOIN review_ranked AS r
        ON d.order_id = r.order_id
       AND r.rn = 1
)

SELECT
    delay_group,
    COUNT(*) AS reviewed_orders,
    ROUND(AVG(review_score), 2) AS avg_score,
    ROUND(
        100.0 * SUM(
            CASE WHEN review_score <= 2
                 THEN 1 ELSE 0 END
        ) / COUNT(*),
        2
    ) AS low_score_pct
FROM grouped_orders
GROUP BY delay_group
ORDER BY delay_group;
"""

delay_score_analysis = pd.read_sql_query(sql, conn)

display(delay_score_analysis)


,delay_group,reviewed_orders,avg_score,low_score_pct
0,0_on_time,89443,4.29,9.27
1,1_1-3_days,1852,3.29,32.13
2,2_4-7_days,1748,2.10,67.68
3,3_8-14_days,1446,1.67,80.15
4,4_15+_days,1335,1.72,78.35


1、即使只延迟1—3天，低评分比例也明显更高,客户评价可能对配送是否超过预计日期比较敏感。
2、延迟4天及以上的订单尤其值得关注
3、从准时送达到延迟8—14天，客户评分表现总体明显恶化；延迟更久后，低评分比例没有继续上升，而是维持在较高水平。

### 4.5 分析不同州的配送延迟情况

In [14]:
sql = """
WITH delivery_by_state AS (
    SELECT
        c.customer_state,

        CAST(
            JULIANDAY(DATE(o.order_delivered_customer_date))
            - JULIANDAY(DATE(o.order_estimated_delivery_date))
            AS INTEGER
        ) AS days_late

    FROM orders AS o

    JOIN customers AS c
        ON o.customer_id = c.customer_id

    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp IS NOT NULL
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
      AND JULIANDAY(o.order_delivered_customer_date)
          >= JULIANDAY(o.order_purchase_timestamp)
)

SELECT
    customer_state,

    COUNT(*) AS total_orders,

    SUM(
        CASE WHEN days_late > 0
             THEN 1 ELSE 0 END
    ) AS late_orders,

    ROUND(
        100.0 * SUM(
            CASE WHEN days_late > 0
                 THEN 1 ELSE 0 END
        ) / COUNT(*), 2
    ) AS late_rate_pct,

    SUM(
        CASE WHEN days_late >= 4
             THEN 1 ELSE 0 END
    ) AS severe_late_orders

FROM delivery_by_state

GROUP BY customer_state

ORDER BY severe_late_orders DESC

LIMIT 10;
"""

state_delivery = pd.read_sql_query(sql, conn)

display(state_delivery)


,customer_state,total_orders,late_orders,late_rate_pct,severe_late_orders
0,RJ,12350,1495,12.11,1198
1,SP,40494,1820,4.49,1128
2,MG,11354,519,4.57,339
3,BA,3256,396,12.16,303
4,RS,5344,325,6.08,242
5,SC,3546,291,8.21,200
6,ES,1995,214,10.73,158
7,CE,1279,176,13.76,146
8,PE,1593,153,9.60,124
9,PR,4923,199,4.04,123


地区履约分析发现： 不同客户收货地区的配送延迟表现存在明显差异。
其中，RJ的延迟率为12.11%，高于平台整体6.77%的水平，且涉及1,495笔延迟订单。
考虑延迟发生比例与受影响订单规模，RJ可作为后续履约优化的重点调查对象。

### 4.6 首次订单评分与后续90天复购的关联分析

In [15]:
sql = """
WITH delivered_orders AS (
    SELECT
        o.order_id,
        c.customer_unique_id,
        o.order_purchase_timestamp AS purchase_time
    FROM orders AS o
    JOIN customers AS c
        ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp IS NOT NULL
),

first_orders AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY customer_unique_id
            ORDER BY purchase_time, order_id
        ) AS rn
    FROM delivered_orders
),

first_reviews AS (
    SELECT
        order_id,
        review_score,
        review_answer_timestamp AS review_time,
        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY review_answer_timestamp DESC,
                     review_id DESC
        ) AS rn
    FROM reviews
    WHERE review_score BETWEEN 1 AND 5
      AND review_answer_timestamp IS NOT NULL
),

eligible_customers AS (
    SELECT
        f.customer_unique_id,
        f.order_id AS first_order_id,
        r.review_score,
        r.review_time
    FROM first_orders AS f
    JOIN first_reviews AS r
        ON f.order_id = r.order_id
       AND r.rn = 1
    WHERE f.rn = 1
      AND DATETIME(r.review_time) >= DATETIME(f.purchase_time)
      AND DATETIME(r.review_time) <= (
          SELECT DATETIME(MAX(purchase_time), '-90 days')
          FROM delivered_orders
      )
),

customer_repeat AS (
    SELECT
        e.customer_unique_id,
        e.review_score,

        MAX(
            CASE
                WHEN d.order_id IS NOT NULL
                THEN 1 ELSE 0
            END
        ) AS repeat_90d

    FROM eligible_customers AS e

    LEFT JOIN delivered_orders AS d
        ON e.customer_unique_id = d.customer_unique_id
       AND d.order_id <> e.first_order_id
       AND DATETIME(d.purchase_time) > DATETIME(e.review_time)
       AND DATETIME(d.purchase_time)
           <= DATETIME(e.review_time, '+90 days')

    GROUP BY e.customer_unique_id, e.review_score
)

SELECT
    CASE
        WHEN review_score <= 2 THEN '1_low_score'
        WHEN review_score = 3 THEN '2_mid_score'
        ELSE '3_high_score'
    END AS review_group,

    COUNT(*) AS customer_count,

    SUM(repeat_90d) AS repeat_customers,

    ROUND(AVG(repeat_90d) * 100, 2)
        AS repeat_rate_90d_pct

FROM customer_repeat

GROUP BY review_group
ORDER BY review_group;
"""

review_repeat = pd.read_sql_query(sql, conn)

display(review_repeat)


,review_group,customer_count,repeat_customers,repeat_rate_90d_pct
0,1_low_score,9755,79,0.81
1,2_mid_score,6182,60,0.97
2,3_high_score,56315,616,1.09


### 4.7 评分群体与90天复购的卡方独立性检验

In [16]:
from scipy.stats import chi2_contingency

# 构建三组客户的复购/未复购人数表
contingency = review_repeat[
    ["repeat_customers"]
].copy()

contingency["not_repeat"] = (
    review_repeat["customer_count"]
    - review_repeat["repeat_customers"]
)

# 卡方独立性检验
chi2, p_value, dof, expected = chi2_contingency(contingency)

print(f"卡方统计量：{chi2:.4f}")
print(f"P值：{p_value:.4f}")
print(f"自由度：{dof}")

if p_value < 0.05:
    print("三个评分群体的复购比例存在统计显著差异")
else:
    print("未发现三个评分群体的复购比例存在统计显著差异")


卡方统计量：6.8477
P值：0.0326
自由度：2
三个评分群体的复购比例存在统计显著差异


**统计解释与局限性：** 通过卡方独立性检验，P 值约为 0.033，说明评分分组与后续 90 天复购存在统计关联。但高、低评分客户的复购率仅相差 0.28 个百分点，实际差异较小；商品类型、配送体验等其他因素可能同时影响两者，因此不能据此推断因果关系。

## 5. 业务总结